# final notebook

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from model_optimizer import optimize_models, create_feature_sets
import time
import warnings
import soccerdata as sd

## read each season data

In [ ]:
data_merged_gw_2016_17 = pd.read_csv('data/2016-17/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2017_18 = pd.read_csv('data/2017-18/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2018_19 = pd.read_csv('data/2018-19/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2019_20 = pd.read_csv('data/2019-20/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2020_21 = pd.read_csv('data/2020-21/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2021_22 = pd.read_csv('data/2021-22/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2022_23 = pd.read_csv('data/2022-23/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2023_24 = pd.read_csv('data/2023-24/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2024_25 = pd.read_csv('data/2024-25/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2025_26 = pd.read_csv('data/2025-26/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')

### note here that currently I am only making the actions not the analysis I did before to know what to do

## adding position column 

In [ ]:
# Add position column to merged gameweek data based on element_type from player raw data
def add_position_to_merged_gw(merged_gw_df, cleaned_players_df):
    # Map element_type to position names
    element_type_to_position = {
        1: 'Goalkeeper',
        2: 'Defender',
        3: 'Midfielder',
        4: 'Forward'
    }
    # create a new column 'position' in cleaned players dataframe
    cleaned_players_df['position'] = cleaned_players_df['element_type'].map(element_type_to_position)
    # create a mapping from player id to position
    player_id_to_position = dict(zip(cleaned_players_df['id'], cleaned_players_df['position']))
    # add the position column to the merged gw dataframe
    merged_gw_df['position'] = merged_gw_df['element'].map(player_id_to_position)
    return merged_gw_df


### load the players tables for the target seasons

In [ ]:
# first load the raw player data for each season until 2019-20
data_players_2016_17 = pd.read_csv('data/2016-17/players_raw.csv', encoding='latin-1', on_bad_lines='skip')
data_players_2017_18 = pd.read_csv('data/2017-18/players_raw.csv', encoding='latin-1', on_bad_lines='skip')
data_players_2018_19 = pd.read_csv('data/2018-19/players_raw.csv', encoding='latin-1', on_bad_lines='skip')
data_players_2019_20 = pd.read_csv('data/2019-20/players_raw.csv', encoding='latin-1', on_bad_lines='skip')

## add the position column

In [ ]:
# add the position column to each season's player data
data_merged_gw_2016_17 = add_position_to_merged_gw(data_merged_gw_2016_17, data_players_2016_17)
data_merged_gw_2017_18 = add_position_to_merged_gw(data_merged_gw_2017_18, data_players_2017_18)
data_merged_gw_2018_19 = add_position_to_merged_gw(data_merged_gw_2018_19, data_players_2018_19)
data_merged_gw_2019_20 = add_position_to_merged_gw(data_merged_gw_2019_20, data_players_2019_20)

## adding team name to the dataset

In [ ]:
# Add team names to seasons 2016-17 through 2019-20 using master team list
# Process:
# 1. Load master team list (contains season → team_id → team_name mapping)
# 2. Map player_id → team_id from player raw data
# 3. Map team_id → team_name from master list
# 4. Add team_name column to merged GW data
data_master_team_list = pd.read_csv('data/master_team_list.csv', encoding='latin-1', on_bad_lines='skip')
def add_team_name_to_merged_gw(merged_gw_df, players_raw_df, master_team_list_df, season):
    # filter the master team list for the given season
    season_team_list = master_team_list_df[master_team_list_df['season'] == season]
    # create a mapping from team id to team name
    team_id_to_name = dict(zip(season_team_list['team'], season_team_list['team_name']))
    # create a mapping from player id to team id
    player_id_to_team_id = dict(zip(players_raw_df['id'], players_raw_df['team']))
    # create a mapping from player id to team name
    player_id_to_team_name = {player_id: team_id_to_name.get(team_id, 'Unknown') for player_id, team_id in player_id_to_team_id.items()}

    # add the team name column to the merged gw dataframe    return merged_gw_df
    merged_gw_df['team'] = merged_gw_df['element'].map(player_id_to_team_name)

In [ ]:
# apply the function to each season's merged gw data
add_team_name_to_merged_gw(data_merged_gw_2016_17, data_players_2016_17, data_master_team_list, '2016-17')
add_team_name_to_merged_gw(data_merged_gw_2017_18, data_players_2017_18, data_master_team_list, '2017-18')
add_team_name_to_merged_gw(data_merged_gw_2018_19, data_players_2018_19, data_master_team_list, '2018-19')
add_team_name_to_merged_gw(data_merged_gw_2019_20, data_players_2019_20, data_master_team_list, '2019-20')

# ensuring all the seasons has the same attributes

In [ ]:
# Drop the xP (expected points) column from seasons 2020-21 through 2025-26 for consistency
data_merged_gw_2020_21 = data_merged_gw_2020_21.drop(columns=['xP'])
data_merged_gw_2021_22 = data_merged_gw_2021_22.drop(columns=['xP'])
data_merged_gw_2022_23 = data_merged_gw_2022_23.drop(columns=['xP'])
data_merged_gw_2023_24 = data_merged_gw_2023_24.drop(columns=['xP'])
data_merged_gw_2024_25 = data_merged_gw_2024_25.drop(columns=['xP'])
data_merged_gw_2025_26 = data_merged_gw_2025_26.drop(columns=['xP'])
# Drop columns not available across all seasons (2016-17 to 2018-19)
data_merged_gw_2016_17 = data_merged_gw_2016_17.drop(columns=['attempted_passes','big_chances_created','big_chances_missed','completed_passes','dribbles','ea_index','errors_leading_to_goal','errors_leading_to_goal_attempt','fouls','key_passes','kickoff_time_formatted','loaned_in','loaned_out','offside','open_play_crosses','penalties_conceded','tackled','target_missed','winning_goals'])
data_merged_gw_2017_18 = data_merged_gw_2017_18.drop(columns=['attempted_passes','big_chances_created','big_chances_missed','completed_passes','dribbles','ea_index','errors_leading_to_goal','errors_leading_to_goal_attempt','fouls','key_passes','kickoff_time_formatted','loaned_in','loaned_out','offside','open_play_crosses','penalties_conceded','tackled','target_missed','winning_goals'])
data_merged_gw_2018_19 = data_merged_gw_2018_19.drop(columns=['attempted_passes','big_chances_created','big_chances_missed','completed_passes','dribbles','ea_index','errors_leading_to_goal','errors_leading_to_goal_attempt','fouls','key_passes','kickoff_time_formatted','loaned_in','loaned_out','offside','open_play_crosses','penalties_conceded','tackled','target_missed','winning_goals'])
# Drop from 2022 the xp and that stuff
data_merged_gw_2022_23 = data_merged_gw_2022_23.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
data_merged_gw_2023_24 = data_merged_gw_2023_24.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
data_merged_gw_2024_25 = data_merged_gw_2024_25.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
data_merged_gw_2025_26 = data_merged_gw_2025_26.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
# Drop modified in the last two seasons
data_merged_gw_2024_25 = data_merged_gw_2024_25.drop(columns=['modified'])
data_merged_gw_2025_26 = data_merged_gw_2025_26.drop(columns=['modified'])
# drop the id from the first two seasons
data_merged_gw_2016_17 = data_merged_gw_2016_17.drop(columns=['id'])
data_merged_gw_2017_18 = data_merged_gw_2017_18.drop(columns=['id'])
data_merged_gw_2018_19 = data_merged_gw_2018_19.drop(columns=['id'])

## adding defensive contribution

In [ ]:
# Calculate defensive_contribution for seasons 2016-17 to 2024-25
# FPL scoring rules:
# - Defenders: clearances_blocks_interceptions + tackles
# - Midfielders/Forwards: clearances_blocks_interceptions + tackles + recoveries
def add_defensive_contribution(merged_gw_df):
    def calculate_defensive_contribution(row):
        position = row['position']
        clearances = row.get('clearances_blocks_interceptions', 0)
        tackles = row.get('tackles', 0)
        recoveries = row.get('recoveries', 0)
        if position == 'Defender':
            return clearances + tackles
        elif position in ['Midfielder', 'Forward']:
            return clearances + tackles + recoveries
        else:
            return 0
    merged_gw_df['defensive_contribution'] = merged_gw_df.apply(calculate_defensive_contribution, axis=1)
    return merged_gw_df

In [ ]:
# Add placeholder columns (value 0) for defensive stats not tracked in seasons 2019-20 to 2024-25
# This ensures consistent schema across all seasons before calculating defensive_contribution
data_merged_gw_2019_20['clearances_blocks_interceptions'] = 0
data_merged_gw_2019_20['recoveries'] = 0
data_merged_gw_2019_20['tackles'] = 0
data_merged_gw_2020_21['clearances_blocks_interceptions'] = 0
data_merged_gw_2020_21['recoveries'] = 0
data_merged_gw_2020_21['tackles'] = 0
data_merged_gw_2021_22['clearances_blocks_interceptions'] = 0
data_merged_gw_2021_22['recoveries'] = 0
data_merged_gw_2021_22['tackles'] = 0
data_merged_gw_2022_23['clearances_blocks_interceptions'] = 0
data_merged_gw_2022_23['recoveries'] = 0
data_merged_gw_2022_23['tackles'] = 0
data_merged_gw_2023_24['clearances_blocks_interceptions'] = 0
data_merged_gw_2023_24['recoveries'] = 0
data_merged_gw_2023_24['tackles'] = 0
data_merged_gw_2024_25['clearances_blocks_interceptions'] = 0
data_merged_gw_2024_25['recoveries'] = 0
data_merged_gw_2024_25['tackles'] = 0
data_merged_gw_2016_17 = add_defensive_contribution(data_merged_gw_2016_17)
data_merged_gw_2017_18 = add_defensive_contribution(data_merged_gw_2017_18)
data_merged_gw_2018_19 = add_defensive_contribution(data_merged_gw_2018_19)
data_merged_gw_2019_20 = add_defensive_contribution(data_merged_gw_2019_20)
data_merged_gw_2020_21 = add_defensive_contribution(data_merged_gw_2020_21)
data_merged_gw_2021_22 = add_defensive_contribution(data_merged_gw_2021_22)
data_merged_gw_2022_23 = add_defensive_contribution(data_merged_gw_2022_23)
data_merged_gw_2023_24 = add_defensive_contribution(data_merged_gw_2023_24)
data_merged_gw_2024_25 = add_defensive_contribution(data_merged_gw_2024_25)

### verify that all the cols are identical now

In [ ]:
# compare the columns of all these datasets
merged_2016_17_columns = set(data_merged_gw_2016_17.columns.tolist())
merged_2017_18_columns = set(data_merged_gw_2017_18.columns.tolist())
merged_2018_19_columns = set(data_merged_gw_2018_19.columns.tolist())
merged_2019_20_columns = set(data_merged_gw_2019_20.columns.tolist())
merged_2020_21_columns = set(data_merged_gw_2020_21.columns.tolist())
merged_2021_22_columns = set(data_merged_gw_2021_22.columns.tolist())
merged_2022_23_columns = set(data_merged_gw_2022_23.columns.tolist())
merged_2023_24_columns = set(data_merged_gw_2023_24.columns.tolist())
merged_2024_25_columns = set(data_merged_gw_2024_25.columns.tolist())
merged_2025_26_columns = set(data_merged_gw_2025_26.columns.tolist())
# find the common columns across all seasons
common_merged_columns_all_seasons = merged_2016_17_columns.intersection(merged_2017_18_columns).intersection(merged_2018_19_columns).intersection(merged_2019_20_columns).intersection(merged_2020_21_columns).intersection(merged_2021_22_columns).intersection(merged_2022_23_columns).intersection(merged_2023_24_columns).intersection(merged_2024_25_columns).intersection(merged_2025_26_columns)
print("Common Columns Across All Seasons:", sorted(common_merged_columns_all_seasons))
# find the unique columns in each season compared to the common columns
unique_2016_17_columns = merged_2016_17_columns - common_merged_columns_all_seasons
unique_2017_18_columns = merged_2017_18_columns - common_merged_columns_all_seasons
unique_2018_19_columns = merged_2018_19_columns - common_merged_columns_all_seasons
unique_2019_20_columns = merged_2019_20_columns - common_merged_columns_all_seasons
unique_2020_21_columns = merged_2020_21_columns - common_merged_columns_all_seasons
unique_2021_22_columns = merged_2021_22_columns - common_merged_columns_all_seasons
unique_2022_23_columns = merged_2022_23_columns - common_merged_columns_all_seasons
unique_2023_24_columns = merged_2023_24_columns - common_merged_columns_all_seasons
unique_2024_25_columns = merged_2024_25_columns - common_merged_columns_all_seasons
unique_2025_26_columns = merged_2025_26_columns - common_merged_columns_all_seasons
print("Unique Columns in 2016-17:", sorted(unique_2016_17_columns))
print("Unique Columns in 2017-18:", sorted(unique_2017_18_columns))
print("Unique Columns in 2018-19:", sorted(unique_2018_19_columns))
print("Unique Columns in 2019-20:", sorted(unique_2019_20_columns))
print("Unique Columns in 2020-21:", sorted(unique_2020_21_columns))
print("Unique Columns in 2021-22:", sorted(unique_2021_22_columns))
print("Unique Columns in 2022-23:", sorted(unique_2022_23_columns))
print("Unique Columns in 2023-24:", sorted(unique_2023_24_columns))
print("Unique Columns in 2024-25:", sorted(unique_2024_25_columns))
print("Unique Columns in 2025-26:", sorted(unique_2025_26_columns))


# hsitorical points adjustment

## here there is a key decision: 
### the points of the previous seasons will be modified to work with the same system as the new rules (adding points for defensive contribution)

### consider moving this to the end (after merging with the defensive data)

1. Standardize position values across all seasons2. Adjust points for seasons 2016-17 to 2018-19 to match current FPL defensive contribution scoring

In [ ]:
# Adjust historical points (2016-17 to 2018-19) to align with current FPL scoring system
# Add 2 bonus points when:
# - Defenders reach defensive_contribution >= 10
# - Midfielders/Forwards reach defensive_contribution >= 12
def modify_points(merged_gw_df):
    def calculate_modified_points(row):
        points = row['total_points']
        position = row['position']
        defensive_contribution = row['defensive_contribution']
        if position == 'Defender' and defensive_contribution >= 10:
            points += 2
        elif position in ['Midfielder', 'Forward'] and defensive_contribution >= 12:
            points += 2
        return points
    merged_gw_df['total_points'] = merged_gw_df.apply(calculate_modified_points, axis=1)
    return merged_gw_df
data_merged_gw_2016_17 = modify_points(data_merged_gw_2016_17)
data_merged_gw_2017_18 = modify_points(data_merged_gw_2017_18)
data_merged_gw_2018_19 = modify_points(data_merged_gw_2018_19)

### note here that you need to run modify points of the seasons from 2019 to 2025 when you merge with the defensive data

# merging all seasons data in a single dataset

In [ ]:
# Add season identifier to each dataset before merging
data_merged_gw_2016_17['season'] = '2016-17'
data_merged_gw_2017_18['season'] = '2017-18'
data_merged_gw_2018_19['season'] = '2018-19'
data_merged_gw_2019_20['season'] = '2019-20'
data_merged_gw_2020_21['season'] = '2020-21'
data_merged_gw_2021_22['season'] = '2021-22'
data_merged_gw_2022_23['season'] = '2022-23'
data_merged_gw_2023_24['season'] = '2023-24'
data_merged_gw_2024_25['season'] = '2024-25'
data_merged_gw_2025_26['season'] = '2025-26'
# concatenating all seasons data into a single dataframe
all_seasons_data = pd.concat([data_merged_gw_2016_17, data_merged_gw_2017_18, data_merged_gw_2018_19, data_merged_gw_2019_20, data_merged_gw_2020_21, data_merged_gw_2021_22, data_merged_gw_2022_23, data_merged_gw_2023_24, data_merged_gw_2024_25, data_merged_gw_2025_26], ignore_index=True)
print("All Seasons Data Sample:")
print(all_seasons_data.sample(10))

## standarizing the position across seasons

In [ ]:
# Standardize position values to short codes for consistency
all_seasons_data['position'] = all_seasons_data['position'].replace({'Goalkeeper': 'GK', 'Defender': 'DEF', 'Midfielder': 'MID', 'Forward': 'FWD'})

## converting the opponent team from id to name

In [ ]:
# Convert opponent_team from team IDs to team names using master team list
# Create season-specific mappings of team_id -> team_name
team_id_name_mapping = {}
for season in all_seasons_data['season'].unique():
    season_team_data = data_master_team_list[data_master_team_list['season'] == season]
    team_id_name_mapping[season] = dict(zip(season_team_data['team'], season_team_data['team_name']))
# replace the opponent_team in all_seasons_data based on the season and the team id
def replace_opponent_team(row):
    season = row['season']
    team_id = row['opponent_team']
    return team_id_name_mapping[season].get(team_id, team_id)
all_seasons_data['opponent_team'] = all_seasons_data.apply(replace_opponent_team, axis=1)

# saving the current state of data as csv

In [ ]:
# save all seasons data to a csv file
all_seasons_data.to_csv('data/all_seasons_2016_17_to_2025_26/merged_gw_all_seasons.csv', index=False, encoding='latin-1')

# here we add the game number feature then we make the merging with defensive stats